In [ ]:
import concurrent
import importlib
import matplotlib.pyplot as plt
import pandas as pd
import pickle
import random
import sys
from tqdm.auto import tqdm

# performance imports for torch: torch kernel uses one core only.
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["TORCH_NUM_THREADS"] = "1" 

import torch

sys.path.insert(0, '..')
sys.path.insert(0, '../..')
sys.path.insert(0, '../../..')
sys.path.insert(0, '../../../..')
sys.path.insert(0, '../../../../..')
sys.path.insert(0, '../../../../../..')

# Decision labeling runtime models
import data_processing.decision_labeling
importlib.reload(data_processing.decision_labeling)
from data_processing.decision_labeling import DecisionLabeler

# Model
import suffix_pred.models.K_UED_LSTM
importlib.reload(suffix_pred.models.K_UED_LSTM)
from suffix_pred.models.K_UED_LSTM import DropoutUncertaintyEncoderDecoderLSTM

# Decision-rule guided decoding and reasoning
import suffix_pred.decision_rule_guided_reasoning_inference
importlib.reload(suffix_pred.decision_rule_guided_reasoning_inference)
from suffix_pred.decision_rule_guided_reasoning_inference import DecisionGuidanceConfig, get_decision_guided_evaluator

# Evaluation metrics
import suffix_pred.evalaution.evalaution_metrics
importlib.reload(suffix_pred.evalaution.evalaution_metrics)
from suffix_pred.evalaution.evalaution_metrics import evaluate_dls, dls_per_prefix_length, average_dls

In [ ]:
# Model
# Model clean:
file_path_model = '../../../../../../models/Sepsis/clean/Sepsis_UED_LSTM_v1_clean.pkl'

# Model decision_train:
# file_path_model = '../../../../../../models/Sepsis/decision/Sepsis_UED_LSTM_v1_DA.pkl'

model = DropoutUncertaintyEncoderDecoderLSTM.load(file_path_model, dropout=0.1)

# Load the data: Use for eval always the clean test data:
file_path_test = '../../../../../../data/Sepsis/tensor_data/normal/sepsis_all_5_test.pkl'
test_dataset = torch.load(file_path_test, weights_only=False)

# Decision mining artifacts for runtime decision guidance
petri_net_path = '../../../../../../data/Sepsis/Petri_net/sepsis.pkl'
decision_bundle_path = '../../../../../../data/Sepsis/Petri_net/data_aware_Petri_net/decision_places_bundle.json'
decision_model_dir = '../../../../../../data/Sepsis/Petri_net/data_aware_Petri_net/models'

with open(petri_net_path, 'rb') as f:
    net, im, fm = pickle.load(f)

# Attributes must match decision model
dynamic_attributes = ["org:group",
                      "lifecycle:transition",
                      "case_elapsed_time",
                      "event_elapsed_time",
                      "Leucocytes",
                      "CRP",
                      "LacticAcid"]

# Same across the whole trace
static_attributes = ["Age",
                     "InfectionSuspected",
                     "Diagnose",
                     "DiagnosticLacticAcid",
                     "DiagnosticBlood",
                     "DiagnosticArtAstrup",
                     "DiagnosticIC",
                     "DiagnosticSputum",
                     "DiagnosticLiquor",
                     "DiagnosticOther",
                     "DiagnosticUrinarySediment",
                     "DiagnosticECG",
                     "DiagnosticUrinaryCulture",
                     "DiagnosticXthorax",
                     "SIRSCritTachypnea",
                     "SIRSCritHeartRate",
                     "SIRSCriteria2OrMore",
                     "SIRSCritTemperature",
                     "SIRSCritLeucos",
                     "Hypotensie",
                     "Oligurie",
                     "Infusion",
                     "Hypoxie",
                     "DisfuncOrg"]

decision_labeler = DecisionLabeler(petri_net=(net, im, fm),
                                   decision_model_dir=decision_model_dir,
                                   decision_places_bundle_path=decision_bundle_path,
                                   dynamic_attributes=dynamic_attributes,
                                   static_attributes=static_attributes)

guidance_cfg = DecisionGuidanceConfig(epsilon=1e-3,
                                      beta_max=2.0,
                                      alpha=0.10,
                                      support_threshold=0.05)

print('Model and decision guidance artifacts loaded.')

In [ ]:

# Step 1: Guided probabilistic decode suffixes (MC-SA with local decision reweighting)
samples_per_case = 100
worker_count = 32
random_order = False

# Process-local decoder instance for parallel probabilistic inference
_GUIDED_PROB_DECODER = None

def _default_reasoning_row():
    return {'decision_steps': 0,
            'conflicts': 0,
            'conflict_rate': 0.0,
            'explained_steps': 0,
            'explained_rate': 0.0,
            'trace': []}

def _align_sample_reasonings(decoded_suffixes, reasonings):
    aligned = list(reasonings) if reasonings is not None else []
    needed = len(decoded_suffixes)
    while len(aligned) < needed:
        aligned.append(_default_reasoning_row())
    return aligned[:needed]

def _aggregate_prob_reasonings(reasonings):
    if len(reasonings) == 0:
        return _default_reasoning_row()

    decision_steps = sum(r.get('decision_steps', 0) for r in reasonings)
    conflicts = sum(r.get('conflicts', 0) for r in reasonings)
    explained_steps = sum(r.get('explained_steps', 0) for r in reasonings)
    conflict_rate = (conflicts / decision_steps) if decision_steps > 0 else 0.0
    explained_rate = (explained_steps / decision_steps) if decision_steps > 0 else 0.0

    # Keep one representative trace for qualitative inspection
    trace_example = []
    for r in reasonings:
        trace_example = r.get('trace', [])
        if len(trace_example) > 0:
            break

    return {'decision_steps': int(decision_steps),
            'conflicts': int(conflicts),
            'conflict_rate': float(conflict_rate),
            'explained_steps': int(explained_steps),
            'explained_rate': float(explained_rate),
            'trace': trace_example}

def _init_guided_prob_worker(model,
                             dataset,
                             petri_net_path,
                             decision_bundle_path,
                             decision_model_dir,
                             dynamic_attributes,
                             static_attributes,
                             guidance_kwargs,
                             samples_per_case):
    global _GUIDED_PROB_DECODER

    with open(petri_net_path, 'rb') as f:
        net_w, im_w, fm_w = pickle.load(f)

    decision_labeler_w = DecisionLabeler(petri_net=(net_w, im_w, fm_w),
                                         decision_model_dir=decision_model_dir,
                                         decision_places_bundle_path=decision_bundle_path,
                                         dynamic_attributes=dynamic_attributes,
                                         static_attributes=static_attributes)

    guidance_cfg_w = DecisionGuidanceConfig(**guidance_kwargs)

    _GUIDED_PROB_DECODER = get_decision_guided_evaluator(kind='mcsa',
                                                         model=model,
                                                         dataset=dataset,
                                                         decision_labeler=decision_labeler_w,
                                                         guidance_config=guidance_cfg_w,
                                                         decision_places_bundle_path=decision_bundle_path,
                                                         concept_name='concept:name',
                                                         eos_value='EOS',
                                                         samples_per_case=int(samples_per_case),
                                                         sample_argmax=False,
                                                         use_variance_cat=True,
                                                         variational_dropout_sampling=False)

def _collect_guided_prob_case_chunk(case_ids):
    if _GUIDED_PROB_DECODER is None:
        raise RuntimeError('Guided probabilistic worker decoder is not initialized.')

    decoder = _GUIDED_PROB_DECODER
    rows = []
    reason_rows = []

    for case_id in case_ids:
        full_case = decoder.cases.get(case_id)
        if full_case is None:
            continue

        for prefix_len, prefix, zero_mask, statics, suffix in decoder._iterate_case(full_case):
            prefix_activity = decoder._decode_activity_prefix(prefix)
            target_suffix = decoder._decode_activity_suffix(suffix)

            sampled_suffixes = []
            reasonings = []

            for _ in range(decoder.samples_per_case):
                sampled, reasoning = decoder.sample_suffix(prefix=prefix,
                                                          prefix_len=prefix_len,
                                                          static_inputs=statics,
                                                          mask=zero_mask,
                                                          suffix=suffix,
                                                          include_model_states=False,
                                                          return_reasoning=True)
                sampled_suffixes.append(sampled)
                reasonings.append(reasoning)

            aligned_reasonings = _align_sample_reasonings(sampled_suffixes, reasonings)

            rows.append({'case_id': case_id,
                         'prefix_len': int(prefix_len),
                         'prefix': prefix_activity,
                         'target_suffix': target_suffix,
                         'decoded_suffixes': sampled_suffixes,
                         'mode': 'guided_probabilistic'})

            reason_rows.append({'case_id': case_id,
                                'prefix_len': int(prefix_len),
                                # Keep every sampled reasoning (1:1 with sampled_suffixes).
                                'reasonings': aligned_reasonings,
                                # Keep aggregate alias for backward compatibility.
                                'reasoning': _aggregate_prob_reasonings(aligned_reasonings)})

    return rows, reason_rows

# Build a local decoder once to enumerate case ids
guided_prob_eval = get_decision_guided_evaluator(kind='mcsa',
                                                 model=model,
                                                 dataset=test_dataset,
                                                 decision_labeler=decision_labeler,
                                                 guidance_config=guidance_cfg,
                                                 decision_places_bundle_path=decision_bundle_path,
                                                 concept_name='concept:name',
                                                 eos_value='EOS',
                                                 samples_per_case=samples_per_case,
                                                 sample_argmax=False,
                                                 use_variance_cat=True,
                                                 variational_dropout_sampling=False)

case_ids = list(guided_prob_eval.cases.keys())
if random_order:
    random.shuffle(case_ids)

guided_prob_outputs = []
guided_prob_reasoning = []

if worker_count <= 1 or len(case_ids) <= 1:
    for case_id in tqdm(case_ids, desc='Guided probabilistic inference cases'):
        full_case = guided_prob_eval.cases.get(case_id)
        if full_case is None:
            continue

        for prefix_len, prefix, zero_mask, statics, suffix in guided_prob_eval._iterate_case(full_case):
            prefix_activity = guided_prob_eval._decode_activity_prefix(prefix)
            target_suffix = guided_prob_eval._decode_activity_suffix(suffix)

            sampled_suffixes = []
            reasonings = []

            for _ in range(guided_prob_eval.samples_per_case):
                sampled, reasoning = guided_prob_eval.sample_suffix(prefix=prefix,
                                                                   prefix_len=prefix_len,
                                                                   static_inputs=statics,
                                                                   mask=zero_mask,
                                                                   suffix=suffix,
                                                                   include_model_states=False,
                                                                   return_reasoning=True)
                sampled_suffixes.append(sampled)
                reasonings.append(reasoning)

            aligned_reasonings = _align_sample_reasonings(sampled_suffixes, reasonings)

            guided_prob_outputs.append({'case_id': case_id,
                                        'prefix_len': int(prefix_len),
                                        'prefix': prefix_activity,
                                        'target_suffix': target_suffix,
                                        'decoded_suffixes': sampled_suffixes,
                                        'mode': 'guided_probabilistic'})

            guided_prob_reasoning.append({'case_id': case_id,
                                          'prefix_len': int(prefix_len),
                                          'reasonings': aligned_reasonings,
                                          'reasoning': _aggregate_prob_reasonings(aligned_reasonings)})
else:
    chunk_size = max(1, (len(case_ids) + worker_count - 1) // worker_count)
    case_chunks = [case_ids[i:i + chunk_size] for i in range(0, len(case_ids), chunk_size)]

    guidance_kwargs = {'epsilon': 1e-3, 'beta_max': 2.0, 'alpha': 0.10, 'support_threshold': 0.05}

    with concurrent.futures.ProcessPoolExecutor(
        max_workers=worker_count,
        initializer=_init_guided_prob_worker,
        initargs=(model,
                  test_dataset,
                  petri_net_path,
                  decision_bundle_path,
                  decision_model_dir,
                  dynamic_attributes,
                  static_attributes,
                  guidance_kwargs,
                  samples_per_case),
    ) as executor:
        chunk_futures = [(i, executor.submit(_collect_guided_prob_case_chunk, chunk))
                         for i, chunk in enumerate(case_chunks)]
        pending = {f: i for i, f in chunk_futures}
        chunk_results = {}
        for future in tqdm(concurrent.futures.as_completed(pending), total=len(pending), desc='Guided probabilistic inference chunks'):
            chunk_results[pending[future]] = future.result()

    # Reassemble in original submission order so rows match C-LSTM / T-GAN case order
    for i in range(len(case_chunks)):
        chunk_outputs, chunk_reasoning = chunk_results[i]
        guided_prob_outputs.extend(chunk_outputs)
        guided_prob_reasoning.extend(chunk_reasoning)

cache_outputs_path = '../../../../../../eval_results/Sepsis/decision_decoding/sepsis_ued_lstm_decision_guided_outputs.pkl'
cache_reasoning_path = '../../../../../../eval_results/Sepsis/decision_decoding/sepsis_ued_lstm_decision_guided_reasoning.pkl'

with open(cache_outputs_path, 'wb') as f:
    pickle.dump(guided_prob_outputs, f)

with open(cache_reasoning_path, 'wb') as f:
    pickle.dump(guided_prob_reasoning, f)

print(f"Decoded {len(guided_prob_outputs)} prefix rows in 'guided_probabilistic'.")

# Step 2: Evaluate DLS from guided decoded outputs
guided_prob_df = evaluate_dls(guided_prob_outputs, probabilistic_reduction='mean', parallel_scoring=True, num_processes=worker_count)
guided_prob_per_prefix = dls_per_prefix_length(guided_prob_df)
guided_prob_avg = average_dls(guided_prob_df)
guided_prob_min = float(guided_prob_df['dls_min'].mean())
guided_prob_max = float(guided_prob_df['dls_max'].mean())

# Step 3: Aggregate decision and explanation diagnostics across all samples
decision_steps = sum(rr.get('decision_steps', 0)
                    for row in guided_prob_reasoning
                    for rr in row.get('reasonings', []))
conflicts = sum(rr.get('conflicts', 0)
                for row in guided_prob_reasoning
                for rr in row.get('reasonings', []))
explained_steps = sum(rr.get('explained_steps', 0)
                      for row in guided_prob_reasoning
                      for rr in row.get('reasonings', []))

conflict_rate = (conflicts / decision_steps) if decision_steps > 0 else 0.0
explained_rate = (explained_steps / decision_steps) if decision_steps > 0 else 0.0

print(f"Average DLS (guided_probabilistic, mean over T samples): {guided_prob_avg:.4f}")
print(f"Average DLS min (guided_probabilistic): {guided_prob_min:.4f}")
print(f"Average DLS max (guided_probabilistic): {guided_prob_max:.4f}")
print(f"Decision steps: {decision_steps} | Conflicts: {conflicts} | Conflict rate: {conflict_rate:.4f}")
print(f"Explained steps: {explained_steps} | Explained rate: {explained_rate:.4f}")


In [ ]:
# Plot DLS by prefix length for guided probabilistic decoding
plt.figure(figsize=(11, 6))

plt.plot(guided_prob_per_prefix['prefix_len'],
         guided_prob_per_prefix['dls'],
         marker='o',
         label=f"guided_probabilistic mean (avg={guided_prob_avg:.3f})")

if {'dls_min', 'dls_max'}.issubset(guided_prob_per_prefix.columns):
    plt.fill_between(guided_prob_per_prefix['prefix_len'],
                     guided_prob_per_prefix['dls_min'],
                     guided_prob_per_prefix['dls_max'],
                     alpha=0.2,
                     label='guided_probabilistic min-max range')

plt.title('Sepsis K-UED-LSTM: Activity DLS by Prefix Length (guided probabilistic decoding)')
plt.xlabel('Prefix length')
plt.ylabel('Damerau-Levenshtein Similarity (DLS)')
plt.ylim(0.0, 1.0)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    'mode': ['guided_probabilistic'],
    'average_dls': [guided_prob_avg],
    'average_dls_min': [guided_prob_min],
    'average_dls_max': [guided_prob_max],
    'decision_steps': [decision_steps],
    'conflicts': [conflicts],
    'conflict_rate': [conflict_rate],
    'explained_steps': [explained_steps],
    'explained_rate': [explained_rate],
}).sort_values('average_dls', ascending=False).reset_index(drop=True)
summary

In [ ]:
# Inspect guided probabilistic predictions and rule-based reasoning examples
with open('../../../../../../eval_results/Sepsis/decision_decoding/sepsis_ued_lstm_decision_guided_outputs.pkl', 'rb') as f:
    cached_outputs = pickle.load(f)
    
with open('../../../../../../eval_results/Sepsis/decision_decoding/sepsis_ued_lstm_decision_guided_reasoning.pkl', 'rb') as f:
    cached_reasoning = pickle.load(f)

def _default_reasoning_row():
    return {'decision_steps': 0,
            'conflicts': 0,
            'conflict_rate': 0.0,
            'explained_steps': 0,
            'explained_rate': 0.0,
            'trace': []}

for i in range(min(100, len(cached_outputs))):
    row = cached_outputs[i]
    reason_row = cached_reasoning[i]

    sample_reasonings = reason_row.get('reasonings', [])
    if len(sample_reasonings) == 0 and reason_row.get('reasoning', None) is not None:
        # Backward compatibility with previous cache format.
        sample_reasonings = [reason_row['reasoning']]

    print(f"Case: {row['case_id']}  |  Prefix len: {row['prefix_len']}")
    print(f"  Prefix:            {row['prefix']}")
    print(f"  Target suffix:     {row['target_suffix']}")
    decoded = row['decoded_suffixes']
    print(f"  Sampled suffixes:  {len(decoded)}")

    for j, sample in enumerate(decoded[:3]):
        reason = sample_reasonings[j] if j < len(sample_reasonings) else _default_reasoning_row()

        print(f"  Sample {j}: {sample}")
        print(f"    Decision steps: {reason['decision_steps']} | Conflicts: {reason['conflicts']} | Conflict rate: {reason['conflict_rate']:.4f}")
        print(f"    Explained steps: {reason.get('explained_steps', 0)} | Explained rate: {reason.get('explained_rate', 0.0):.4f}")

        trace = reason.get('trace', [])
        if len(trace) == 0:
            print("    No supported rule-aligned reasoning step.")
            continue

        for step in trace[:3]:
            decision_top = step.get('decision_top_event')
            decision_top_prob = step.get('decision_top_prob')
            next_event = step['next_event']
            model_prob = step.get('model_prob')
            model_prob_str = f"{model_prob:.1%}" if model_prob is not None else "?"

            if decision_top is not None and decision_top != next_event:
                # Decision model's top pick differs from what the suffix predictor chose
                top_prob_str = f"p={decision_top_prob:.1%}" if decision_top_prob is not None else "p=?"
                print(f"    Decision step {step['step']}: {step['input_event']} -> "
                      f"{decision_top} ({top_prob_str}, decision top) at place {step['place']}, "
                      f"but with prob {model_prob_str} (model), {next_event} was predicted")
            else:
                top_prob_str = f", decision top p={decision_top_prob:.1%}" if decision_top_prob is not None else ""
                print(f"    Decision step {step['step']}: {step['input_event']} -> "
                      f"{next_event} (model prob: {model_prob_str}{top_prob_str}) at place {step['place']}")

            matched_rule = step.get('matched_rule')
            if matched_rule is not None:
                print(f"      Matched rule: {matched_rule.get('rule', '')}")

            attr_checks = step.get('attribute_checks', [])
            seen_attrs = set()
            if len(attr_checks) == 0:
                print("      No attribute checks available.")
            else:
                for chk in attr_checks:
                    attr_name = chk.get('attr', '')
                    attr_value = chk.get('value', None)
                    dedup_key = (attr_name, attr_value)
                    if dedup_key in seen_attrs:
                        continue
                    seen_attrs.add(dedup_key)
                    is_in_rule = bool(chk.get('in_rule_set', False))
                    print(f"      - {attr_name}: value={attr_value} | in_rule_set={is_in_rule}")
    print()